# Optunaによるパラメータのオートチューニング

In [1]:
!pip install -qq optuna kaggle-environments

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.7/721.7 kB 14.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 944.3/944.3 kB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.3/96.3 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 89.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 840.2/840.2 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.4/111.4 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.4/267.4 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 69.1 MB

In [2]:
"""Colabで実行するパラメータ探索コード。"""

import importlib
import statistics

import optuna
from kaggle_environments import make

import main as strategy


# Colab上の最新のmain.pyを読み込む。
strategy = importlib.reload(strategy)

N_TRIALS = 100
MATCH_COUNT = 20
EVALUATION_SEEDS = list(range(MATCH_COUNT))


def objective(trial):
    """平均得点を返す。"""

    #担当エリア外の作業候補に与える減点
    strategy.StrategyConfig.OUTSIDE_AREA_PENALTY = trial.suggest_int(
        "OUTSIDE_AREA_PENALTY",
        0,
        60,
        step=5,
    )


    rewards = []

    for episode_seed in EVALUATION_SEEDS:
        strategy.hire_controller = strategy.HireController()

        env = make(
            "kaggriculture",
            configuration={
                "episodeSteps": 720,
                "seed": episode_seed,
            },
            debug=True,
        )

        env.run([strategy.agent, strategy.agent])

        for state in env.steps[-1]:
            rewards.append(float(state.reward))

    return statistics.fmean(rewards)


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
)

study.optimize(
    objective,
    n_trials=N_TRIALS,
    n_jobs=1,
    show_progress_bar=True,
)

print("\n===== 最高平均得点 =====")
print(study.best_value)

print("\n===== main.pyへ手動設定する値 =====")

for name, value in study.best_params.items():
    print(f"{name} = {value}")

[I 2026-09-08 12:26:08,747] A new study created in memory with name: no-name-ba2abc0c-483d-42ca-a89c-42b1b0a51714


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-09-08 12:28:55,795] Trial 0 finished with value: 64377.75 and parameters: {'OUTSIDE_AREA_PENALTY': 20}. Best is trial 0 with value: 64377.75.
[I 2026-09-08 12:31:35,797] Trial 1 finished with value: 66030.725 and parameters: {'OUTSIDE_AREA_PENALTY': 60}. Best is trial 1 with value: 66030.725.
[I 2026-09-08 12:34:17,655] Trial 2 finished with value: 67055.975 and parameters: {'OUTSIDE_AREA_PENALTY': 45}. Best is trial 2 with value: 67055.975.
[I 2026-09-08 12:36:58,920] Trial 3 finished with value: 68066.375 and parameters: {'OUTSIDE_AREA_PENALTY': 35}. Best is trial 3 with value: 68066.375.
[I 2026-09-08 12:39:42,208] Trial 4 finished with value: 64949.925 and parameters: {'OUTSIDE_AREA_PENALTY': 10}. Best is trial 3 with value: 68066.375.
[I 2026-09-08 12:42:24,181] Trial 5 finished with value: 64949.925 and parameters: {'OUTSIDE_AREA_PENALTY': 10}. Best is trial 3 with value: 68066.375.
[I 2026-09-08 12:45:09,669] Trial 6 finished with value: 63851.675 and parameters: {'OUTSI